# Séance 6 — Pipeline Data Science

**Decision problem:** can this analysis be rerun reliably when the decision comes back next month?

Official topic preserved: data science pipeline. The output is a validation report and artifact manifest.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]:
    d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"

def load_raw_trends():
    p = DATA / "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
    df = pd.read_csv(p)
    df["date"] = pd.to_datetime(df["date"])
    for c in ["chatgpt", "iphone", "meteo"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)

def load_clean_long():
    p = OUT / "bloc1" / "clean_trends_long.csv"
    if p.exists():
        df = pd.read_csv(p, parse_dates=["date"])
    else:
        raw = load_raw_trends()
        df = raw.melt(id_vars="date", var_name="signal", value_name="interest")
    return df.sort_values(["date", "signal"]).reset_index(drop=True)

In [ ]:
expected = ["clean_trends_long.csv", "eda_signal_summary.csv", "feature_table.csv", "model_metrics.csv", "attention_regimes.csv"]
rows=[]
for name in expected:
    p = OUT / "bloc1" / name
    rows.append({"artifact": name, "exists": p.exists(), "bytes": p.stat().st_size if p.exists() else 0})
report = pd.DataFrame(rows)
report["ready"] = report["exists"] & (report["bytes"] > 0)
report.to_csv(OUT / "bloc1" / "pipeline_validation_report.csv", index=False)
manifest = {"bloc":"bloc1", "ready": bool(report["ready"].all()), "artifacts": rows}
(OUT / "bloc1" / "artifact_manifest.json").write_text(json.dumps(manifest, indent=2))
report

## Practical exercise

Add one artifact that would make the pipeline easier to audit.

## Conclusion

A repeatable pipeline turns classroom analysis into an operational decision process.